In [44]:
from datasets import load_dataset

dataset = load_dataset("galileo-ai/ragbench", "emanual", split="validation")

print(dataset.column_names)

['id', 'question', 'documents', 'response', 'generation_model_name', 'annotating_model_name', 'dataset_name', 'documents_sentences', 'response_sentences', 'sentence_support_information', 'unsupported_response_sentence_keys', 'adherence_score', 'overall_supported_explanation', 'relevance_explanation', 'all_relevant_sentence_keys', 'all_utilized_sentence_keys', 'trulens_groundedness', 'trulens_context_relevance', 'ragas_faithfulness', 'ragas_context_relevance', 'gpt3_adherence', 'gpt3_context_relevance', 'gpt35_utilization', 'relevance_score', 'utilization_score', 'completeness_score']


In [45]:
filtered = dataset.filter(lambda example: example["generation_model_name"] == "gpt-3.5-turbo-0125")

In [46]:
all_docs = []

for item in filtered:
    docs = item.get("documents", [])
    
    for d in docs:

        if isinstance(d, str):
            all_docs.append(d)
        elif isinstance(d, dict):
            text = d.get("text") or d.get("page_content") or d.get("content")
            if text:
                all_docs.append(text)

print(f"Total documents extracted: {len(all_docs)}")


Total documents extracted: 198


In [47]:
from eval_utils import load_llm, load_embeddings, load_docs

# load chat model
chat_model = load_llm(model_path="./hf_models/Llama-3.1-8B-Instruct")

# load embedding model
embedding = load_embeddings(model_id="sentence-transformers/all-MiniLM-L6-v2",
                            cache_dir="hf_models/embeddings/all-MiniLM-L6-v2")

Loading checkpoint shards: 100%|██████████| 4/4 [00:03<00:00,  1.29it/s]
Device set to use cuda:0


In [48]:
from langchain.schema import Document
docs = [Document(page_content=text) for text in all_docs]

In [49]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=256, 
                                               chunk_overlap=32,
                                               separators=["\n## ","\n### ","\n#### ", "\n\n", ".", "\n", " "],
                                               is_separator_regex=False,
                                               keep_separator=False,)
chunks = text_splitter.split_documents(docs)

In [50]:
import time
from typing import List, Set
import pandas as pd
from langchain_community.vectorstores import FAISS
from langchain_core.retrievers import BaseRetriever
from langchain.retrievers.ensemble import EnsembleRetriever
from langchain_community.retrievers import BM25Retriever
from langchain_core.prompts import PromptTemplate
from langchain.retrievers.multi_query import MultiQueryRetriever
from langchain_core.runnables import RunnableLambda
from langchain_core.callbacks import CallbackManagerForRetrieverRun
from langchain_core.documents import Document


vector_storage = FAISS.from_documents(chunks, embedding)

bm25_retriever = BM25Retriever.from_documents(chunks)
bm25_retriever.k = 10

faiss_retriever = vector_storage.as_retriever(
                                search_type="mmr",
                                search_kwargs={"k": 10, "lambda_mult": 0.7}
                                )
class HybridRetriever(BaseRetriever):
    
    base_retriever: BaseRetriever

    def _get_relevant_documents(
        self,
        query: str,
        *,
        run_manager: CallbackManagerForRetrieverRun,
    ) -> List[Document]:
        
        docs = self.base_retriever.get_relevant_documents(query)

        # deduplication
        seen: Set[str] = set()
        unique_docs: List[Document] = []
        for d in docs:
            if d.page_content not in seen:
                seen.add(d.page_content)
                unique_docs.append(d)

        return unique_docs
    
ensemble_retriever = EnsembleRetriever(
        retrievers=[bm25_retriever, faiss_retriever], weights=[0.4, 0.6]
    )

hybrid_retriever = HybridRetriever(base_retriever=ensemble_retriever)

In [51]:
from langchain.prompts import PromptTemplate, FewShotPromptTemplate

#  few-shot template
mq_prompt = PromptTemplate(
    input_variables=["question", "q1", "q2", "q3"],
    template=(
        "You are a helpful research assistant.\n"
        "Your task is to rewrite the user's question into EXACTLY THREE alternative search queries.\n"
        "The queries must be clear, and capture different aspects of the same information need.\n"
        "User question: {question}\n"
        "Provide the 3 reformulated search queries, each on a new line:\n"
    ),
)


# Create MultiQueryRetriever，prompt and LLM
mq_retriever = MultiQueryRetriever.from_llm(
    retriever=hybrid_retriever,
    llm=chat_model,
    prompt=mq_prompt,
    include_original=True,
)

In [54]:
# test output queries
question = 'Can I turn off the TV using off timer?'
generated_queries = mq_retriever.llm_chain.invoke({"question": question})
print("=== Generated Query List ===")
for i, q in enumerate(generated_queries, 1):
    print(f"{i}. {q}")

print("\n=== Running retrieval ===")
docs = mq_retriever.get_relevant_documents(question)

for i, doc in enumerate(docs[:3], 1):
    print(f"Doc {i}: {doc.metadata if hasattr(doc, 'metadata') else ''}")
    print(doc.page_content[:200], "...\n")

=== Generated Query List ===
1. Here are three alternative search queries:
2. 1. How to use the TV's off timer feature.
3. 2. Is it possible to turn off a TV using a scheduled timer.
4. 3. What is the function of a TV's timer for turning off the device.

=== Running retrieval ===
Doc 1: {}
When the background update is completed, it is applied the next time the TV is turned on. If you agree to the Smart Hub terms and conditions, Auto Update is set to automatically. If you want this func ...

Doc 2: {}
You can set the sleep timer for up to 180 minutes after which it will turn off the TV. Turning off the TV using the off timer Settings General System Manager Time Off Timer You can set Off Timer to sh ...

Doc 3: {}
Using the timers. Using the sleep timer Settings General System Manager Time Sleep Timer You can use this function to automatically shut off the TV after a pre-set period of time ...



In [55]:
from langchain.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain_community.document_transformers import EmbeddingsRedundantFilter
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import DocumentCompressorPipeline

reranker_model = HuggingFaceCrossEncoder(model_name="hf_models/rerank/bge-reranker-large",
                                         model_kwargs={"device": "cuda:0"},)
redundant_filter = EmbeddingsRedundantFilter(embeddings=embedding, similarity_threshold=0.9)


reranker  = CrossEncoderReranker(model=reranker_model,top_n=15)

pipeline_compressor = DocumentCompressorPipeline(
        transformers=[redundant_filter,reranker]
    )

compression_retriever = ContextualCompressionRetriever( 
        base_compressor=pipeline_compressor,
        base_retriever=mq_retriever
    )

In [56]:
compression_retriever.invoke("Can I turn off the TV using off timer?")

[_DocumentWithState(metadata={}, page_content='You can set the sleep timer for up to 180 minutes after which it will turn off the TV. Turning off the TV using the off timer Settings General System Manager Time Off Timer You can set Off Timer to shut off the TV automatically at a specific time', state={'embedded_doc': [0.027051961049437523, 0.03770137578248978, 0.024160198867321014, -0.045997437089681625, -0.01608232967555523, 0.08774115145206451, -0.055525343865156174, 0.030075594782829285, 0.032224010676145554, 0.0007218262180685997, -0.05673437938094139, -0.003182163229212165, 0.008811051957309246, 0.010655620135366917, -0.03744715452194214, -0.01747620292007923, 0.01816943660378456, -0.07439202070236206, 0.0353936143219471, 0.057426609098911285, 0.10789534449577332, -0.013696584850549698, -0.0012177102034911513, 0.070780448615551, 0.0008620977168902755, -0.022777272388339043, 0.07913661748170853, -0.0167709831148386, -0.017944911494851112, 0.02431601472198963, 0.033552054315805435, 

In [59]:
import pandas as pd
from langchain_core.runnables import RunnableLambda,RunnablePassthrough
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from langchain.schema.output_parser import StrOutputParser

template = """ 
You are an expert assistant that answers technical questions by using provided retrieved context excerpts.
Only use information from the provided context strictly. If the context does not contain the answer, respond with “I don’t know”.
Do not hallucinate or add unsupported information.

Anwser the question in a clear, concise way.

Question: {question}
Context: {context}
Answer:
"""
prompt = PromptTemplate.from_template(template)

def format_docs(docs):
    return "\n\n".join([d.page_content for d in docs])

#  Build the RAG + reranker chain
rag_chain = (
    {
        "context": compression_retriever| RunnableLambda(format_docs),
        "question": RunnablePassthrough()
    }
    | prompt
    | chat_model
    | StrOutputParser()
)


In [60]:
rag_chain.invoke("Can I turn off the TV using off timer??")

'Yes, you can turn off the TV using the off timer.'

In [61]:
import pandas as pd
import time
from ragas import EvaluationDataset

def get_eval_ds(path, retriever, rag_chain):

    testset = pd.read_csv(path)
    queries = testset["user_input"].to_list()
    references = testset["reference"].to_list()

    # generate evaluation dataset
    rs_times = []
    dataset = []

    for query, reference in zip(queries, references):

        
        relevant_docs = [doc.page_content for doc in retriever.invoke(query)]
        # measure response time
        t0 = time.perf_counter()
        response = rag_chain.invoke(query)
        t1 = time.perf_counter()
        
        dataset.append(
            {
                "user_input": query,
                "retrieved_contexts": relevant_docs,
                "response": response,
                "reference": reference,
            }
        )
        rs_times.append(t1 -t0)

    eval_ds = EvaluationDataset.from_list(dataset)

    return eval_ds, rs_times

from ragas import evaluate, RunConfig
from ragas.metrics import LLMContextPrecisionWithReference, LLMContextRecall, AnswerAccuracy, AnswerRelevancy, Faithfulness, FactualCorrectness
from eval_utils import load_eval_model

def get_eval_result(eval_ds, metrics):
    evaluator_llm = load_eval_model()

    run_config = RunConfig(
    timeout=600,      
    max_workers=2,    
    max_retries=2,    
    max_wait=600,     
)
    result = evaluate(
        dataset=eval_ds,
        metrics=metrics,
        llm=evaluator_llm,
        run_config=run_config,
        )
    
    return result

metrics=[LLMContextPrecisionWithReference(),
         LLMContextRecall(), 
         AnswerAccuracy(),
         AnswerRelevancy(),
         Faithfulness(),
         FactualCorrectness(mode = 'f1', atomicity='high', coverage='high')]

In [63]:
from dotenv import load_dotenv
load_dotenv()

data_path = "evaluate_dataset/test_dataset_ragbench.csv"

eval_ds, response_time = get_eval_ds(data_path, compression_retriever, rag_chain)
len(eval_ds)


66

In [64]:
result = get_eval_result(eval_ds, metrics)
result

Evaluating: 100%|██████████| 396/396 [45:35<00:00,  6.91s/it]  


{'llm_context_precision_with_reference': 0.7031, 'context_recall': 0.8178, 'nv_accuracy': 0.3636, 'answer_relevancy': 0.7361, 'faithfulness': 0.6229, 'factual_correctness(mode=f1)': 0.3203}

In [65]:
df = result.to_pandas()
df["response_time"] = response_time
df.to_csv(f"./evaluate_results/benchmark.csv")